In [0]:
spark.sql("DROP VOLUME IF EXISTS employeedataplatform.bronze.raw_data")

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import glob
import urllib.request
from pyspark.sql import DataFrame
from datetime import date

# ---------- Config ----------
config = {
    "source": {
        "gdrive_file_ids": {
            "DimCancellationReason.csv": "1DlttKk_V8E7FlH_d_h7RDmyZz6vM9z3P",
            "DimCounty.csv": "1zZMcSbGoCfT373UyKdlAGp1mYHkjeNPR",
            "DimPeople.csv": "1pjA84d5kpaehxuWcnmKyldc5EEWdwTp_",
            "DimShrinkageType.csv": "17fcUCses3anWnOpN984_cdyBxa1IGQF7",
            "FactShift_01_04_2026.csv": "1UAkXgtYu-Dfm3ZF1-5kDrJfmuxFb_F5w",
            "FactShrinkage_01_04_2026.csv": "1MsUb9JaMRmSVH1px_KorJW9sXlEZjQYI",
            "FactWork_01_04_2026.csv": "1PIryJ6CC-90gFt1eZebmGwbvwUr75x6d"
        },
        "file_format": "csv"
    },
    "target": {
        "base_path" : "/Volumes/employeedataplatform/bronze/raw_data",
        "catalog": "employeedataplatform",
        "schema": "bronze",
        "volume": "raw_data",
        "destination_folder": "01_04_2026",
        "file_format": "csv",
        "mode": "overwrite",
        "num_output_files": 1
    },
    "read_options": {
        "header": "true",
        "inferSchema": "false",
        "mode": "PERMISSIVE"
    },
    "write_options": {
        "header": "true",
        "quoteAll": "true",
        "escape": "\""
    }
}

import urllib.request
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# ---------- Config ----------
GDRIVE_FILE_ID = config["source"]["gdrive_file_ids"]
GDRIVE_FILE_FORMAT = config["source"]["file_format"]

CATALOG = config["target"]["catalog"]
SCHEMA = config["target"]["schema"]
VOLUME = config["target"]["volume"]

base_path = config["target"]["base_path"]
destination_folder = config["target"]["destination_folder"]

run_date = date.today().isoformat()
output_path = f"{base_path}/load_date={run_date}/{destination_folder}"


# ---------- Functions ----------
def setup_storage():
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
    dbutils.fs.mkdirs(output_path)


def download_from_gdrive(file_ids: dict, folder_path: str):
    for file_name, file_id in file_ids.items():
        url = f"https://drive.google.com/uc?export=download&id={file_id}"
        urllib.request.urlretrieve(url, f"{folder_path}/{file_name}")

# ---------- Ru ----------
setup_storage()
download_from_gdrive(GDRIVE_FILE_ID, output_path)

In [0]:
import urllib.request
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# ---------- Config ----------
GDRIVE_FILE_ID = "1CYAmF1BzVzpxnsHXD5scjzGTuK_cIcEh"

CATALOG = "employeedataplatform"
SCHEMA = "bronze"
VOLUME = "raw_data"

output_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/load_date=26-10-2026"
output_file = f"{output_path}/DimShrinkageType.csv"


# ---------- Functions ----------
def setup_storage():
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
    dbutils.fs.mkdirs(output_path)


def download_from_gdrive(file_id: str, output_path: str):
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    urllib.request.urlretrieve(url, output_path)


def read_csv(path: str) -> DataFrame:
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("mode", "PERMISSIVE")
        .csv(path)
    )


def write_to_delta(df: DataFrame, table_name: str):
    (
        df.write
            .mode("overwrite")
            .format("csv")
            .option("header", True)
            .option("quoteAll", True)
            .option("escape", "\"")
            .save(output_path)
    )


# ---------- Run ----------
setup_storage()
download_from_gdrive(GDRIVE_FILE_ID, output_file)
employees_df = read_csv(output_file)
row_count = employees_df.count()
print(f"Loaded {row_count} rows into {output_path}")


In [0]:
# ---------- Config ----------
GDRIVE_FILE_ID = "1sYNaHDbrRXv3wQRO9qIq0IrZJe1vaXLy"

CATALOG = "employeedataplatform"
SCHEMA = "bronze"
VOLUME = "raw_data"

output_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/load_date=26-10-2026"
output_file = f"{output_path}/DimPeople.csv"


# ---------- Functions ----------
def setup_storage():
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
    dbutils.fs.mkdirs(output_path)


def download_from_gdrive(file_id: str, output_path: str):
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    urllib.request.urlretrieve(url, output_path)


def read_csv(path: str) -> DataFrame:
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("mode", "PERMISSIVE")
        .csv(path)
    )


def write_to_delta(df: DataFrame, table_name: str):
    (
        df.write
            .mode("overwrite")
            .format("csv")
            .option("header", True)
            .option("quoteAll", True)
            .option("escape", "\"")
            .save(output_path)
    )


# ---------- Run ----------
setup_storage()
download_from_gdrive(GDRIVE_FILE_ID, output_file)
employees_df = read_csv(output_file)
row_count = employees_df.count()
print(f"Loaded {row_count} rows into {output_path}")

In [0]:
# ---------- Config ----------
GDRIVE_FILE_ID = "1y3-arLu8hwd6Gtk9emaJrk47Be_kGmmz"

CATALOG = "employeedataplatform"
SCHEMA = "bronze"
VOLUME = "raw_data"

output_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/load_date=26-10-2026"
output_file = f"{output_path}/DimCounty.csv"


# ---------- Functions ----------
def setup_storage():
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
    dbutils.fs.mkdirs(output_path)


def download_from_gdrive(file_id: str, output_path: str):
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    urllib.request.urlretrieve(url, output_path)


def read_csv(path: str) -> DataFrame:
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("mode", "PERMISSIVE")
        .csv(path)
    )


def write_to_delta(df: DataFrame, table_name: str):
    (
        df.write
            .mode("overwrite")
            .format("csv")
            .option("header", True)
            .option("quoteAll", True)
            .option("escape", "\"")
            .save(output_path)
    )


# ---------- Run ----------
setup_storage()
download_from_gdrive(GDRIVE_FILE_ID, output_file)
employees_df = read_csv(output_file)
row_count = employees_df.count()
print(f"Loaded {row_count} rows into {output_path}")